# Q15-Q16: IRLS for Total Variation Regularization

## Objectives
1. Apply IRLS algorithm to small bag
2. Compare Total Variation vs Tikhonov regularization
3. Reconstruct large bag contents
4. Identify objects in the bag

## Theory
IRLS solves: $\min_x \frac{1}{2}\|Ax - y\|^2 + \alpha\|Lx\|_1$

Key properties of TV regularization:
- Preserves edges (piecewise constant solutions)
- Better for discrete objects
- Non-smooth optimization ($\ell_1$ norm)

IRLS approximates $\ell_1$ via iteratively weighted $\ell_2$ problems

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.append('..')

from src.data_utils import load_3d_data, vector_to_grid, save_reconstruction
from src.matrix_construction import build_combined_derivative_matrix
from src.solvers import cgls, irls
from src.visualization import (
    visualize_3d_slices,
    compare_reconstructions,
    plot_histogram,
    plot_convergence
)

%matplotlib inline

## Q15: Small Bag - Tikhonov vs Total Variation

In [ ]:
# Load small bag data
y_small, A_small = load_3d_data('../data/Small')
n = 19
L = build_combined_derivative_matrix(n, n, dimensions=3)

### Step 1: Solve with Tikhonov (for initialization and comparison)

In [ ]:
# Use best λ from Q11
lam_best = 1e-5

print("Solving with Tikhonov regularization...")
result_tikh = cgls(A_small, y_small, L, lam=lam_best, tol=1e-6, max_iter=300)

print(f"Converged: {result_tikh.converged}")
print(f"Iterations: {result_tikh.iterations}")
print(f"Final objective: {result_tikh.objective_values[-1]:.6e}")

X_tikh = vector_to_grid(result_tikh.x, (n, n, n))

### Step 2: Solve with IRLS (Total Variation)

In [ ]:
# Test different α values
alpha_values = [0.1, 0.5, 1.0]

for alpha in alpha_values:
    print(f"\n{'='*60}")
    print(f"Solving with IRLS (α={alpha})...")
    print('='*60)
    
    result_tv = irls(
        A_small, y_small, L,
        alpha=alpha,
        x0=result_tikh.x,  # Initialize with Tikhonov solution
        epsilon=1e-8,
        tol=1e-4,
        max_outer_iter=20,
        max_inner_iter=100,
        verbose=True
    )
    
    X_tv = vector_to_grid(result_tv.x, (n, n, n))
    
    # Visualize slices
    visualize_3d_slices(X_tv, axis='z', num_slices=9,
                       title=f"Small Bag - Total Variation (α={alpha})",
                       save_path=f'../results/figures/small_bag_tv_alpha{alpha}.png')

### Step 3: Compare Tikhonov vs TV (using best α)

In [ ]:
# Choose best alpha
alpha_best = 0.5  # Adjust based on results above

# Re-run with best alpha if needed
result_tv_best = irls(
    A_small, y_small, L,
    alpha=alpha_best,
    x0=result_tikh.x,
    epsilon=1e-8,
    tol=1e-4,
    max_outer_iter=20,
    max_inner_iter=100,
    verbose=True
)

X_tv_best = vector_to_grid(result_tv_best.x, (n, n, n))

In [ ]:
# Side-by-side comparison
slice_idx = n // 2

compare_reconstructions(
    X_tikh, X_tv_best,
    labels=("Tikhonov (ℓ²)", "Total Variation (ℓ¹)"),
    slice_idx=slice_idx,
    axis='z',
    save_path='../results/figures/small_bag_tikhonov_vs_tv.png'
)

### Step 4: Compare Histograms

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.hist(X_tikh.flatten(), bins=50, edgecolor='black', alpha=0.7)
ax1.set_xlabel('Density Value')
ax1.set_ylabel('Frequency')
ax1.set_title('Tikhonov (ℓ²) - Smoother Distribution')
ax1.grid(True, alpha=0.3)

ax2.hist(X_tv_best.flatten(), bins=50, edgecolor='black', alpha=0.7, color='orange')
ax2.set_xlabel('Density Value')
ax2.set_ylabel('Frequency')
ax2.set_title('Total Variation (ℓ¹) - More Peaked Distribution')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../results/figures/small_bag_histogram_comparison.png', dpi=150)
plt.show()

## Qualitative Comparison

**Your analysis here:**

### Edge Preservation
- Tikhonov: 
- Total Variation: 

### Piecewise Constant Regions
- Tikhonov: 
- Total Variation: 

### Which is better for discrete objects?
- 


## Q16: Reconstruct Large Bag

In [ ]:
# Load large bag data
y_large, A_large = load_3d_data('../data/Large')
n_large = 49
print(f"Large bag volume: {n_large}³ = {n_large**3:,} voxels")

In [ ]:
# Build 3D derivative for large volume
L_large = build_combined_derivative_matrix(n_large, n_large, dimensions=3)
print(f"Derivative matrix shape: {L_large.shape}")

### Initialize with Tikhonov

In [ ]:
print("Initializing with Tikhonov (this may take a while...)")
result_large_tikh = cgls(
    A_large, y_large, L_large,
    lam=1e-5,
    tol=1e-6,
    max_iter=300
)

print(f"\nTikhonov completed:")
print(f"  Converged: {result_large_tikh.converged}")
print(f"  Iterations: {result_large_tikh.iterations}")

### Solve with IRLS (Total Variation)

In [ ]:
print("Solving large bag with IRLS (this will take several minutes...)")
result_large_tv = irls(
    A_large, y_large, L_large,
    alpha=0.5,
    x0=result_large_tikh.x,
    epsilon=1e-8,
    tol=1e-4,
    max_outer_iter=20,
    max_inner_iter=200,
    verbose=True
)

print(f"\nIRLS completed:")
print(f"  Converged: {result_large_tv.converged}")
print(f"  Outer iterations: {result_large_tv.iterations}")

In [ ]:
# Convert to 3D grid
X_large = vector_to_grid(result_large_tv.x, (n_large, n_large, n_large))

# Save reconstruction
save_reconstruction(
    result_large_tv.x,
    '../results/reconstructions/large_bag_tv_reconstruction.npz',
    metadata={'alpha': 0.5, 'n': n_large, 'converged': result_large_tv.converged}
)

### Visualize Large Bag Contents

In [ ]:
# Visualize slices along z-axis
visualize_3d_slices(
    X_large,
    axis='z',
    num_slices=12,
    title="Large Bag Contents (z-slices)",
    save_path='../results/figures/large_bag_z_slices.png'
)

In [ ]:
# Visualize slices along x-axis
visualize_3d_slices(
    X_large,
    axis='x',
    num_slices=12,
    title="Large Bag Contents (x-slices)",
    save_path='../results/figures/large_bag_x_slices.png'
)

In [ ]:
# Histogram of density values
plot_histogram(
    X_large.flatten(),
    bins=100,
    title="Large Bag - Density Distribution",
    save_path='../results/figures/large_bag_histogram.png'
)

### Analyze Contents

In [ ]:
# Statistics
print("Density Statistics:")
print(f"  Min: {X_large.min():.6f}")
print(f"  Max: {X_large.max():.6f}")
print(f"  Mean: {X_large.mean():.6f}")
print(f"  Std: {X_large.std():.6f}")
print(f"  Median: {np.median(X_large):.6f}")

# Percentiles
percentiles = [10, 25, 50, 75, 90, 95, 99]
print("\nPercentiles:")
for p in percentiles:
    val = np.percentile(X_large, p)
    print(f"  {p}th: {val:.6f}")

In [ ]:
# Optional: Threshold-based object detection
thresholds = np.percentile(X_large, [75, 85, 95])

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
slice_idx = n_large // 2

for idx, thresh in enumerate(thresholds):
    binary_mask = X_large[:, :, slice_idx] > thresh
    axes[idx].imshow(binary_mask, cmap='binary', origin='lower')
    axes[idx].set_title(f'Threshold at {int(np.percentile(X_large, [75, 85, 95])[idx]/thresh*100)}th percentile')
    axes[idx].axis('off')

plt.tight_layout()
plt.savefig('../results/figures/large_bag_thresholding.png', dpi=150)
plt.show()

## What did you find in the bag?

**Your analysis here:**

### Object 1:
- Location: 
- Shape: 
- Approximate density: 
- Interpretation: 

### Object 2:
- Location: 
- Shape: 
- Approximate density: 
- Interpretation: 

### Suspicious items?
- 
